# Lab 3 — Benchmarking fidelity: does an organoid regulome predict perturbations?

*Third session of the [`notebooks/` course](README.md) on computational synthetic morphology
(after [Lab 2 — Regulomes and hypergraphs](02_regulomes_and_hypergraphs.ipynb)).*

A regulome is a **hypothesis** about causal wiring — "this TF regulates these genes." A hypothesis
earns its keep by predicting: knock a TF out, do the cells move the way the wiring says? This lab
runs that test on the **cerebral-organoid regulome of Fleck et al. (2023, *Nature*)** (720 TFs,
2,792 genes — the same `H` you loaded in Lab 2), against perturbation screens at increasing
distance from where it was built:

- **In-domain** — CROP-seq in the *same* organoid system (Fleck's own screen). *Sketched* here: the
  committed `data/cropseq/*.csv` tables are synthetic, GRN-informed *stand-ins* (the real Seurat
  extraction is `scripts/extract_cropseq.py`; the project's real in-domain agreement, e.g. GLI3-vs-
  TBR1 KO signatures r ≈ 0.83, is in `ROADMAP.md`). We keep it as an exercise (§6a).
- **Cross-system, same species** — primary human cortical CRISPRi, **Ding, Kim *et al.* / Pollen lab
  (2026, *Nature*)** (GEO `GSE284197`; 44 TFs perturbed in 2-D primary cultures). *This is the
  focus.* Does a dish-grown regulome transfer to real cortex?
- **Cross-species** — marmoset / mouse cortical GRNs (§6c).

The headline you'll arrive at: **direction transfers, magnitude does not.** Knocking out a TF in
primary cortex moves its regulome-predicted targets *the right way* most of the time (≈ 83% across
the 34 shared TFs), but the *size* of the move is only weakly predicted (Pearson r ≈ 0.13 zero-shot,
vs ≈ 0.36 for a model fit *within* primary tissue). That gap — and *why* it's there — is the lab,
and it connects straight to the structural-identifiability story of [Lab 7](07_structural_identifiability.ipynb).

**Needs:** `numpy`, `matplotlib`, optionally `scipy` (one hypergeometric test). All inputs are
committed — the Fleck regulome (`data/processed/`), the Pollen primary-cortex DE
(`data/pollen/processed/`), per-cell-type and pattern-fidelity summaries (`figures/pollen_*.json`,
`figures/advanced_fidelity_results.json`), and the organoid CROP-seq stand-ins (`data/cropseq/*.csv`)
— and every cell falls back to a committed number or a tiny synthetic example if something is
missing. The full pipeline behind this lab: `scripts/compare_pollen.py` (the four-way Fleck↔Pollen
benchmark) and `scripts/benchmark_advanced_fidelity.py`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy import stats as _scipy_stats
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

rng = np.random.default_rng(0)

def _find(*parts):
    here = Path.cwd()
    for base in [here, *here.parents]:
        p = base.joinpath(*parts)
        if p.exists():
            return p
    return None

# ---- Fleck organoid regulome: H, plus the in-silico KO predictions ----------------------------
PROC = _find("data", "processed")
HAVE_FLECK = PROC is not None and (PROC / "incidence.npy").exists()
if HAVE_FLECK:
    incidence  = np.load(PROC / "incidence.npy").astype(np.float32)            # (genes, regulons)
    gene_names = json.loads((PROC / "gene_names.json").read_text())
    tf_names   = json.loads((PROC / "tf_names.json").read_text())              # one per regulon
    key_tfs    = json.loads((PROC / "summary.json").read_text())["key_tfs"]    # rows of pert_eff
    pert_eff   = np.load(PROC / "perturbation_effects.npy").astype(np.float32) # (n_key_tf, genes)
    FLECK_SRC  = "Fleck et al. 2023 cerebral-organoid regulome (Pando)"
else:
    print("[note] data/processed/ not found — using a tiny synthetic regulome + KO predictions.")
    gene_names = [f"g{i}" for i in range(60)]
    tf_names   = [f"TF{i}" for i in range(15)]
    key_tfs    = ["TF0", "TF1", "TF2"]
    incidence  = (rng.random((60, 15)) < 0.2).astype(np.float32)
    pert_eff   = np.zeros((3, 60), dtype=np.float32)
    for r, t in enumerate(key_tfs):
        mem = np.where(incidence[:, tf_names.index(t)] > 0)[0]
        pert_eff[r, mem] = rng.normal(size=mem.size)
    FLECK_SRC  = "synthetic toy regulome (60 genes, 15 regulons)"

# ---- Pollen primary-cortex CRISPRi DE: {ko_gene: {gene: log2fc}} ------------------------------
POLL = _find("data", "pollen", "processed")
HAVE_POLLEN = POLL is not None and (POLL / "summary.json").exists()
pollen_de = {}
psum = {}
if HAVE_POLLEN:
    psum = json.loads((POLL / "summary.json").read_text())
    de_csv = POLL / "pollen_de.csv"
    if de_csv.exists():
        import csv
        with open(de_csv) as fh:
            for row in csv.DictReader(fh):
                try:
                    pollen_de.setdefault(row["ko_gene"], {})[row["gene"]] = float(row["log2fc"])
                except (KeyError, ValueError):
                    pass

# ---- organoid CROP-seq DE (the IN-DOMAIN stand-ins; used only in §6a) -------------------------
CROP = _find("data", "cropseq")
cropseq_de, CROPSEQ_SYNTHETIC = {}, True
if CROP is not None and (CROP / "cropseq_summary.json").exists():
    import csv
    csum = json.loads((CROP / "cropseq_summary.json").read_text())
    CROPSEQ_SYNTHETIC = bool(csum.get("is_synthetic", True))
    for ko, meta in csum.get("knockouts", {}).items():
        f = CROP / meta["file"]
        if f.exists():
            with open(f) as fh:
                cropseq_de[ko] = {row["gene"]: float(row["log2fc"])
                                  for row in csv.DictReader(fh) if row.get("log2fc") not in (None, "")}

# ---- precomputed benchmarks (so headline numbers survive a stripped checkout) -----------------
def _loadjson(*parts):
    p = _find(*parts);  return json.loads(p.read_text()) if p else None
pollen_bench   = _loadjson("figures", "pollen_comparison_results.json")
pollen_byct    = _loadjson("figures", "pollen_filtered_results.json")
adv_fidelity   = _loadjson("figures", "advanced_fidelity_results.json")

print(f"Fleck regulome        : {FLECK_SRC}")
print(f"  genes / regulons    : {len(gene_names)} / {len(tf_names)}")
print(f"  in-silico KO preds   : {len(key_tfs)} TFs ({', '.join(key_tfs)})")
print(f"Pollen primary CRISPRi: {'loaded — ' + str(len(pollen_de)) + ' KO DE tables' if pollen_de else 'not present (using committed summary)'}")
print(f"organoid CROP-seq     : {len(cropseq_de)} stand-in DE tables"
      + ("  [⚠ synthetic GRN-informed — see §6a]" if CROPSEQ_SYNTHETIC else "  [real]"))
print(f"precomputed benchmarks: pollen_comparison={bool(pollen_bench)}  pollen_filtered={bool(pollen_byct)}  advanced_fidelity={bool(adv_fidelity)}")

## 1. Three fidelity questions

"Is the regulome any good?" is too vague to test. Three sharper questions, in increasing order of
difficulty:

| # | Question | What you compare | Survives transfer to a new system? |
|---|----------|------------------|------------------------------------|
| 1 | **Overlap** — does the regulome even *talk about* the right TFs? | the organoid regulon TFs vs the TFs people actually screen | yes — it's coverage, not dynamics |
| 2 | **Direction** — when a TF is knocked out, do downstream genes move the *right way*? | sign of predicted ΔE vs sign of measured ΔE, per gene | mostly — ≈ 83% organoid→primary (with a trained predictor; raw regulon signs ≈ 60%) |
| 3 | **Magnitude** — *how much* does each gene move? | predicted ΔE vs measured ΔE (Pearson r) | barely — r ≈ 0.13 zero-shot, ≈ 0.36 within-primary |

A fourth, *operational* question wraps the three: train a perturbation predictor on the organoid
data, test it **zero-shot** on primary tissue, and compare against one fit *within* primary tissue.
The difference is the **domain-shift cost** — the price of using a dish-grown model to reason about
real cortex.

This lab does (1) live, (2)/(3) live on the TFs the regulome ships KO predictions for *and* via the
full 34-TF benchmark, then disaggregates the raw transfer by cell type, and finally separates
"fidelity" from two things it is *not*: parameter **identifiability** (Lab 7) and module
**separability** (Lab 4).

## 2. Overlap — does the organoid regulome talk about the right TFs?

Before asking whether predictions are *right*, ask whether the regulome even *covers* the TFs that
matter. The Pollen lab chose 44 TFs to CRISPRi in primary cortex — presumably the ones a cortical-
development expert thinks are load-bearing. How many of those 44 show up as regulons in the
*organoid* regulome?

If the regulome were a random 720 of the ~1,600 human TFs, you'd expect 44 × 720/1600 ≈ 20 of them
by chance. We test the observed overlap against that null with a one-sided hypergeometric test.

In [ ]:
fleck_tf_set = set(tf_names)
if HAVE_POLLEN and psum.get("tf_list"):
    screened = list(psum["tf_list"])
elif pollen_bench:
    screened = list(pollen_bench["overlap"]["pollen_tfs"])
else:
    screened = ["TBR1", "EMX1", "NEUROD6"]                     # synthetic fallback
shared = sorted(fleck_tf_set & set(screened))
N_HUMAN_TFS = 1600                                             # ~ size of the human TF repertoire

print(f"organoid regulon TFs : {len(fleck_tf_set)}")
print(f"screened (Pollen) TFs: {len(screened)}")
print(f"shared               : {len(shared)}")
print(f"                       {shared}")
exp = len(screened) * len(fleck_tf_set) / N_HUMAN_TFS
print(f"expected by chance   : {exp:.1f}  (of {N_HUMAN_TFS} human TFs)")
if HAVE_SCIPY and shared:
    pval = float(_scipy_stats.hypergeom.sf(len(shared) - 1, N_HUMAN_TFS, len(screened), len(fleck_tf_set)))
elif pollen_bench:
    pval = pollen_bench["overlap"]["enrichment_pval"]
else:
    pval = float("nan")
print(f"hypergeometric p     ≈ {pval:.2e}")

fig, ax = plt.subplots(figsize=(5.8, 3.6))
labels = ["organoid\nregulons", "screened TFs\n(Pollen)", "shared", "expected\nby chance"]
vals   = [len(fleck_tf_set), len(screened), len(shared), exp]
ax.bar(labels, vals, color=["#4C72B0", "#55A868", "#C44E52", "#bbbbbb"])
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.012, f"{v:.0f}" if i != 3 else f"{v:.1f}", ha="center", va="bottom")
ax.set_ylabel("number of TFs")
ax.set_title(f"GRN / screen overlap  —  hypergeometric p ≈ {pval:.1e}")
fig.tight_layout(); plt.show()

## 3. Transfer — direction survives, magnitude does not

Now the hard test. For each TF screened in *primary cortex*, the *organoid* regulome predicts a
signed downstream-effect vector. We correlate it with the measured primary-cortex CRISPRi log2FC.
The regulome ships in-silico KO predictions for `key_tfs`, a few of which (`EMX1`, `NEUROD6`, `TBR1`,
…) Pollen also screened — so we can do this **live** for those — and the **full 34-TF benchmark**
(an `hgx` `PerturbationPredictor` trained on the whole Fleck regulome) is precomputed in
`figures/pollen_comparison_results.json` (`scripts/compare_pollen.py`).

Three numbers to keep apart:

- **Raw direction concordance** — take the regulome's KO-effect vector *as is* (no fitting), check
  the sign against the measurement. This lands near **≈ 60%** — barely above the 50% coin flip. Raw
  regulon signs alone are a *weak* predictor.
- **Trained-predictor direction accuracy** — fit an `hgx` `PerturbationPredictor` on the *whole*
  Fleck regulome, then predict zero-shot. This jumps to **≈ 83%**. The lift from 60→83 is real
  learning (the predictor uses the whole hypergraph, not one regulon's signs), not just the wiring —
  keep that distinction; it's the point of §4.
- **Pearson r (magnitude)** — does it predict the *size*? Zero-shot organoid→primary: r ≈ 0.13. A
  leave-one-out model fit *within* primary tissue: r ≈ 0.36. Even the in-domain ceiling is modest
  (perturbation magnitudes are noisy, context-dependent) and dish→cortex transfer pays a further ~3×.

So the honest summary of *any* regulome's fidelity is a **triple — (in-domain r, transfer direction-
accuracy, transfer r)** — never a single "accuracy". Why is r low even in-domain? Same wall Lab 7
hits symbolically: the regulome's linear surrogate is **not observable/identifiable from the master
regulators alone** — a whole ridge of downstream-magnitude assignments is consistent with what you
measure at the TFs (hence the ridge-shaped SBI posterior of Lab 8). Direction is a *coarse* feature
of that ridge and (with a trained predictor) survives; magnitude is a *fine* feature and washes out.
(The live cell below uses the raw regulome KO vectors directly — so it should land near the ≈ 60%
raw-concordance figure, *not* 83%; that gap is exactly the learning the trained predictor adds.)

In [ ]:
def fidelity(pred, meas, restrict_to_pred_nonzero=True):
    """pred: array aligned to `gene_names`.  meas: {gene: log2fc}.
    -> (n_overlap, pearson_r, direction_accuracy) on the gene overlap."""
    rows = []
    for g, p in zip(gene_names, pred):
        if restrict_to_pred_nonzero and abs(p) < 1e-9:
            continue
        v = meas.get(g)
        if v is not None and np.isfinite(v):
            rows.append((p, v))
    if len(rows) < 3:
        return len(rows), np.nan, np.nan
    P, M = np.array(rows).T
    r = float(np.corrcoef(P, M)[0, 1]) if P.std() > 0 and M.std() > 0 else np.nan
    nz = (P != 0) & (M != 0)
    d = float((np.sign(P[nz]) == np.sign(M[nz])).mean()) if nz.any() else np.nan
    return len(rows), r, d

# --- live zero-shot transfer on the key TFs Pollen also screened -----------------------------
live = [(tf, *fidelity(pert_eff[key_tfs.index(tf)], pollen_de[tf]))
        for tf in key_tfs if tf in pollen_de]
if live:
    print("live zero-shot transfer  (organoid in-silico KO  →  primary-cortex CRISPRi):")
    print(f"  {'TF':>9}  {'n genes':>8}  {'Pearson r':>10}  {'dir.acc':>8}")
    for tf, n, r, d in live:
        print(f"  {tf:>9}  {n:>8d}  {r:>10.3f}  {d:>8.1%}")
    print(f"  → mean over {len(live)} TFs: r = {np.nanmean([x[2] for x in live]):.3f}, "
          f"dir.acc = {np.nanmean([x[3] for x in live]):.1%}  (small N — see the 34-TF benchmark)")
else:
    print("[note] no live overlap available — relying on the committed benchmark below.")

# --- the full four-way benchmark (precomputed) -----------------------------------------------
if pollen_bench:
    co, tr, bl = pollen_bench["concordance"], pollen_bench["transfer"], pollen_bench["baseline"]
    transfer_r, transfer_dacc = tr["mean_pearson_r"], tr["mean_direction_accuracy"]
    baseline_r, raw_concord    = bl["mean_pearson_r"], co["mean_concordance"]
    print("\nfull benchmark (34 shared TFs — scripts/compare_pollen.py):")
    print(f"  raw direction concordance, all shared genes : {raw_concord:.1%}")
    print(f"  zero-shot transfer  — direction accuracy    : {transfer_dacc:.1%}")
    print(f"  zero-shot transfer  — Pearson r             : {transfer_r:.3f}")
    print(f"  within-primary LOO baseline — Pearson r     : {baseline_r:.3f}")
    print(f"  domain-shift cost (baseline_r / transfer_r) : {baseline_r / transfer_r:.1f}×")

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.bar(["transfer\n(zero-shot)", "within-primary\n(LOO baseline)"], [transfer_r, baseline_r],
           color=["#C44E52", "#55A868"])
    for i, v in enumerate([transfer_r, baseline_r]): a1.text(i, v + 0.004, f"{v:.3f}", ha="center")
    a1.set_ylabel("Pearson r  (predicted vs measured ΔE)")
    a1.set_title("Magnitude: ~3× domain-shift penalty")
    a2.bar(["raw\nconcordance", "transfer\ndir.accuracy"], [raw_concord, transfer_dacc],
           color=["#bbbbbb", "#C44E52"]); a2.axhline(0.5, ls="--", lw=.8, color="k")
    for i, v in enumerate([raw_concord, transfer_dacc]): a2.text(i, v + 0.012, f"{v:.1%}", ha="center")
    a2.set_ylim(0, 1); a2.set_ylabel("fraction of genes, correct sign")
    a2.set_title("Direction: survives transfer (chance = 0.5)")
    fig.suptitle("Organoid regulome → primary cortex: the fidelity triple", y=1.03)
    fig.tight_layout(); plt.show()
else:
    print("[note] figures/pollen_comparison_results.json absent — showing live numbers only.")

## 4. The same numbers, per cell type — why "aggregate accuracy" is a trap

`figures/pollen_filtered_results.json` reports the **raw** (no trained predictor) per-gene transfer,
split by annotated cell type — the same raw-concordance measurement as the ≈ 60% above, but not
pooled. Two things to notice:

1. **Per cell type, raw concordance hovers near the coin flip** (≈ 0.4–0.55) and mean r ≈ 0. The raw
   regulome signs, on their own, in any single cell type, are barely informative. The ≈ 83% headline
   of §3 is *the trained `PerturbationPredictor`'s* number — it is doing real work; don't credit it
   to the wiring.
2. **It's also a reminder that "fidelity" aggregated over cell types can hide near-chance behaviour
   inside each.** Always disaggregate before believing a single accuracy.

(If you want the *trained-predictor* number broken down by cell type — which is the fairer "where
does it transfer best" question — that's a `--per-cell-type` run of `scripts/compare_pollen.py`; the
committed JSON here is the raw version, kept small.)

In [ ]:
if pollen_byct:
    print(f"{'cell type':>34}  {'mean r':>9}  {'mean concord':>13}  {'n TFs':>6}")
    rows_ct = []
    for ct, recs in pollen_byct.items():
        rs = np.array([x["r"] for x in recs]); cs = np.array([x["concord"] for x in recs])
        rows_ct.append((ct, float(rs.mean()), float(cs.mean()), len(recs)))
        print(f"{ct:>34}  {rs.mean():>9.3f}  {cs.mean():>12.1%}  {len(recs):>6d}")
    rows_ct.sort(key=lambda x: -x[1])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
    cts = [r[0] for r in rows_ct]
    a1.barh(range(len(cts)), [r[1] for r in rows_ct], color="#4C72B0")
    a1.set_yticks(range(len(cts))); a1.set_yticklabels(cts); a1.invert_yaxis()
    a1.axvline(0, lw=.6, color="k"); a1.set_xlabel("mean Pearson r across shared TFs")
    a1.set_title("Raw magnitude transfer (≈ 0), per cell type")
    a2.barh(range(len(cts)), [r[2] for r in rows_ct], color="#C44E52")
    a2.set_yticks(range(len(cts))); a2.set_yticklabels(cts); a2.invert_yaxis()
    a2.axvline(0.5, ls="--", lw=.8, color="k"); a2.set_xlim(0, 1)
    a2.set_xlabel("mean raw direction concordance"); a2.set_title("Raw direction (≈ coin flip), per cell type")
    fig.suptitle("Raw per-cell-type transfer — the 83% headline is the *trained* predictor, not this", y=1.04)
    fig.tight_layout(); plt.show()
    print("\nNote how the raw, per-cell-type numbers sit near chance — disaggregating kills the"
          "\nillusion of a strong aggregate. The lift to ≈83% comes from learning (§3), not the wiring.")
else:
    print("[note] figures/pollen_filtered_results.json absent — skipping the per-cell-type panel.")

## 5. What "fidelity" is *not*

Three distinctions worth being pedantic about — each a *different* failure mode, tested
*differently*, and conflating them is how people over- or under-sell a regulome:

1. **Fidelity ≠ structural identifiability.** A regulome can predict KO *directions* well while its
   *parameters* (edge weights, Hill exponents in any mechanistic reduction) are unrecoverable even
   from perfect data — the **structural-identifiability** question of
   [Lab 7](07_structural_identifiability.ipynb), decided symbolically *before* any fitting. The
   low transfer-r here is the *practical* (finite-data) shadow of that *structural* fact: the
   downstream-magnitude ridge.
2. **Fidelity ≠ module separability.** A regulome can be a faithful predictor and still *not*
   decompose into clean, distinct modules — the **Module Identifiability Index** (Hodge-Laplacian
   spectral gap) of Lab 4 measures *that*, and it's orthogonal to "does KO(TF) predict the screen".
   (Same word, "identifiability"; three different senses — see `notebooks/README.md`.)
3. **In-domain fidelity ≠ transfer fidelity.** §6a vs §3. A model can ace its training experiment
   and still lose ~3× on the magnitude when moved to a new tissue. Reporting only the in-domain
   number is the single most common way to overstate a regulome.

So when someone says "our regulome is 80% accurate," ask: *80% of what — direction or magnitude? in
the system it was built from, or a new one?* The defensible claim for the Fleck organoid regulome:
"reproduces its own CROP-seq screen; transfers to primary cortex at ≈ 83% direction concordance but
only r ≈ 0.13 on magnitude" — genuinely useful for *qualitative* "which way will this nudge things"
(the anatomical-compiler use case of Lab 8), not enough for quantitative dose prediction.

## 6. Exercises

**(a) Wire in the real in-domain CROP-seq.** The committed `data/cropseq/*.csv` are *synthetic, GRN-
informed* stand-ins (`cropseq_summary.json["is_synthetic"] == True`) and they do **not** share
`pert_eff`'s sign convention — so the cell below mostly produces noise, on purpose, as the *plumbing*
you'd connect real data to. (i) Run it and confirm it's noise. (ii) Then point `cropseq_de` at the
real DE tables produced by `scripts/extract_cropseq.py` (R/Seurat over the Fleck deposited objects)
and re-run — you should recover the in-domain agreement reported in `ROADMAP.md` (e.g. GLI3 vs TBR1
KO signatures, r ≈ 0.83). (iii) Make a scatter for the best-covered KO; restrict to the genes the
regulome makes a claim about (`abs(pert_eff) > 0`) vs all overlapping genes — which is the fairer
comparison, and why?

**(b) Fidelity of *identity*, not perturbation.** `figures/advanced_fidelity_results.json` scores
four systems — organoid (Fleck), bioprinted brain (Tang), hiPSC-NSC (Vassal), bioprinted *liver*
(Zhang) — against three target gene-expression *patterns* (`Mature Neuron`, `Outer Radial Glia`,
`Synaptic (ASD)`). Raw scores aren't comparable across systems (different cell counts) — **normalize
each system's three scores to sum to 1** (the starter does this), then ask: do the *brain*-like
systems put more mass on the neural patterns than the liver does? This is fidelity of *cell identity*
(does the regulome light up the right programs?) vs fidelity of *perturbation* (does KO predict the
screen?) — a good regulome should pass both. (Generator: `scripts/benchmark_advanced_fidelity.py`.)

**(c) Cross-species conservation — and why overlap is the weak test.** The marmoset and mouse
benchmarks (`scripts/benchmark_marmoset.py`, `scripts/benchmark_mouse.py`; `ROADMAP.md` §3E) find
e.g. the **DLX1 regulon conserved** human↔marmoset interneurons (p ≈ 2.1×10⁻³) and **EOMES /
NEUROD2** conserved in mouse neocortex. If you have a non-human GRN handy, recompute the
hypergeometric TF-overlap (§2) between it and the Fleck regulome. Then argue: why is TF-set
*overlap* a far weaker conservation claim than *direction concordance on shared regulons*? (Hint:
the same TFs being "present" says nothing about whether they wire to the same targets — overlap is
the §2 test; conservation needs the §3 test, run cross-species.)

**(d) The honest one-liner.** Using your numbers from §2–§4, write the single sentence you'd put in
a paper's abstract about this regulome's predictive fidelity. Then write the sentence a *critic*
would write. The gap between them is the work left to do.

Starters for (a) and (b):

In [ ]:
# --- Exercise (a) starter: in-domain plumbing (organoid in-silico KO  vs  organoid CROP-seq) ---
if cropseq_de:
    print("in-domain (⚠ ground truth here is a synthetic GRN-informed stand-in — see §6a):")
    print(f"  {'TF':>9}  {'n genes':>8}  {'Pearson r':>10}  {'dir.acc':>8}")
    for tf in key_tfs:
        if tf in cropseq_de:
            n, r, d = fidelity(pert_eff[key_tfs.index(tf)], cropseq_de[tf])
            print(f"  {tf:>9}  {n:>8d}  {r:>10.3f}  {d:>8.1%}")
    print("  ↳ mostly noise, by construction. Swap in scripts/extract_cropseq.py output → real r.")
else:
    print("[note] data/cropseq/ absent — skip (a).")

# --- Exercise (b) starter: normalized pattern-fidelity profiles ----------------------------------
if adv_fidelity:
    datasets = sorted({r["dataset"] for r in adv_fidelity})
    patterns = sorted({r["pattern"] for r in adv_fidelity})
    M = np.array([[next(r["score"] for r in adv_fidelity if r["dataset"] == d and r["pattern"] == p)
                   for p in patterns] for d in datasets], float)
    Mn = M / M.sum(1, keepdims=True)
    print("\nnormalized pattern profile (each row sums to 1):")
    print(f"  {'dataset':>26}  " + "  ".join(f"{p[:18]:>18}" for p in patterns))
    for d, row in zip(datasets, Mn):
        print(f"  {d:>26}  " + "  ".join(f"{v:>18.1%}" for v in row))
    # your turn: bar-plot the profiles; do the brain-like systems load more on the neural patterns?
else:
    print("[note] figures/advanced_fidelity_results.json absent — skip (b).")

## Recap & where this sits

- A regulome's fidelity is a **triple**, not a number: *in-domain* magnitude r, *transfer*
  direction-accuracy, *transfer* magnitude r. For the Fleck organoid regulome: reproduces its own
  CROP-seq screen (§6a, real numbers in `ROADMAP.md`); covers the screened TFs far above chance
  (34/44, p ≈ 10⁻⁵, §2); transfers to primary cortex with **direction ≈ 83%** (with a *trained*
  predictor — raw regulon signs are only ≈ 60%, §4) but **magnitude r ≈ 0.13** vs ≈ 0.36 in-domain
  ceiling (§3). And disaggregating by cell type shows the raw numbers sit near chance — aggregate
  accuracy can lie (§4).
- *Why* magnitude doesn't survive is the **practical** face of [Lab 7](07_structural_identifiability.ipynb)'s
  **structural** result — the downstream-magnitude ridge that makes the SBI posterior (Lab 8) a
  valley, not a point. Fidelity, identifiability (structural), and module separability (Lab 4) are
  three different questions with three different tests — §5.
- **Next:** Lab 4 — Modularity & identifiability (the Hodge Laplacian and the Module Identifiability
  Index — "does this regulome decompose into clean modules?"), then Lab 5 (Hypergraph Neural ODEs —
  *learning* the dynamics rather than reading off KO signs), and Lab 8 (the anatomical compiler —
  where "direction transfers, magnitude doesn't" is exactly why optimal control is posed on the
  *learned* ODE with feedback, not open-loop on the regulome).
- For the GPU end-to-end organoid pipeline (preprocessing → figures → the five biological-validation
  checks → the `hgx`-vs-DHG benchmark) see [`organoid_hgx_colab.ipynb`](organoid_hgx_colab.ipynb);
  for the full Fleck↔Pollen comparison used here, `scripts/compare_pollen.py`.